In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
from langchain.globals import set_debug

In [12]:
MCP_SERVER_URL="http://localhost:8000/sse"
OLLAMA_URL="http://localhost:11434"

In [13]:
###############################################
# Custom magic command to skip cells
# To use:
# %%skip

from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    """Skips the execution of the cell."""
    return

In [14]:
###############################################
# set_debug(True) to see agent processing details

set_debug(False)

In [15]:
client = MultiServerMCPClient(
    {
        "Demo": {
            "url": MCP_SERVER_URL,
            "transport": "sse",
        }
    }
)

In [22]:
tools = await client.get_tools()

print('MCP Tools:\n')
for t in tools: 
    print(f'Tool name: {t.name}')
    print(t.description)
    print('--\n')

MCP Tools:

get_provider_names
returns a list of available providers
--

get_provider_roles
returns a list of available provider roles
--

get_available_booking_slots_for_provider

Returns booking 5 earliest time slots for a provider.

Output is ALWAYS JSON with fields:
  - slot_number (integer, sorted ascending)
  - provider_name (string)
  - time_slot (string, format 'YYYY-MM-DD HH24:MI:SS')

--

book_slot_for_provider

Books a client appointment with a provider at a specific slot number.

RULES:
- Inputs (provider_name and slot_number) may come from free-text user input.
- Only book the slot if the inputs exactly match a valid slot returned by
  get_available_booking_slots_for_provider.
- Ensure that the database is updated
--

cancel_slot_for_provider

--

list_booked_appointments_for_client

Returns list of booked appointments for a client. Note that the list of appointments
can be empty.

Output is ALWAYS a list of JSON documents with fields:
  - client_id (string indicating the 

In [17]:
prompts = await client.get_prompt('Demo', 'configure_assistant')

print('MCP Prompts:\n')
for p in prompts:
    print(p.content)

MCP Prompts:


        You are a scheduling assistant. You must ONLY interact with the environment using the following tools:
        - list_booked_appointments_for_client → returns a list booked appointments for a client id
        - get_provider_names → returns a list of available providers
        - get_provider_roles → returns a list of provider roles
        - get_available_booking_slots_for_provider → returns list of 5 earliest available time slots for a provider. 
          response is JSON with a top-level "result" field.
          "result" is a list of dicts, each with:
            - slot_number (integer, sorted ascending)
            - provider_name (string)
            - time_slot (string in 'YYYY-MM-DD HH24:MI:SS' format).

        - book_slot_for_provider → books a client appointment with a provider at a specified slot number.

        STRICT RULES:
        1. NEVER invent, summarize, analyze, or describe provider names, roles, or time slots. 
        2. NEVER generate you

In [18]:
###############################################
# LLM to use in react agent

llm = ChatOllama(model="qwen3:8b", base_url=OLLAMA_URL, reasoning=True, temperature=0)

In [19]:
###############################################
# react agent creation

agent = create_react_agent(
    llm,
    tools, 
    prompt=prompts[0].content
)

In [ ]:
#####################################################
# List of providers at clinic

resp = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What care providers are available in our faciliy and what are their roles?"}]}
)
print(resp['messages'][-1].content)

In [ ]:
#####################################################
# List of open booking slots for provider at clinic
user_message = {
    "role": "user",
    "content": "I want to book an apointment with Susan Davis ? Find the 5 earliest times."
}

resp = await agent.ainvoke(
    {"messages": resp['messages'] + [user_message]}
)

# Print the agent's last message
print(resp['messages'][-1].content)

In [ ]:
#####################################################
# Book appointment

user_message = {
    "role": "user",
    "content": "Book an appointment with Susan Davis at time slot_number slot 45 for client id 123"
}

resp = await agent.ainvoke(
    {
        "messages": resp['messages'] + [user_message]
    }
)

print(resp['messages'][-1].content)

In [20]:
#####################################################
# List bookings for client id 123

user_message = {
    "role": "user",
    "content": "What are the booked slots for client id 123?"
}

resp = await agent.ainvoke(
    {
        "messages": [user_message]
    }
)

# Print the agent's last message
print(resp['messages'][-1].content)

The client with ID 123 has no booked appointments.


In [ ]:
#####################################################
# List of open booking slots for provider at clinic - (The slot booked should no longer be in this list)

user_message = {
    "role": "user",
    "content": "I want to schedule an apointment with Susan Davis ? Find the 5 earliest times."
}

resp = await agent.ainvoke(
    {
        "messages": [user_message]
    }
)

# Print the agent's last message
print(resp['messages'][-1].content)